In [1]:
## Check the kernel 
import IPython
print(IPython.sys_info())

{'commit_hash': '178fef7',
 'commit_source': 'installation',
 'default_encoding': 'utf-8',
 'ipython_path': 'D:\\Conda Virtual '
                 'Environments\\hiberno-dict\\Lib\\site-packages\\IPython',
 'ipython_version': '9.10.0',
 'os_name': 'nt',
 'platform': 'Windows-10-10.0.26200-SP0',
 'sys_executable': 'd:\\Conda Virtual Environments\\hiberno-dict\\python.exe',
 'sys_platform': 'win32',
 'sys_version': '3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, '
                '17:12:15) [MSC v.1942 64 bit (AMD64)]'}


In [2]:
# =========================================================
# HIBERNO-ENGLISH PARSER NOTEBOOK
# Single-notebook pipeline:
# 1. Read DOCX
# 2. Build paragraph audit table
# 3. Remove obvious headers
# 4. Parse entries
# 5. Save a tracked run automatically
# 6. Compare against previous run automatically
# =========================================================

from pathlib import Path
from datetime import datetime
import re
import json
import hashlib
import collections
import pandas as pd
from docx import Document

# =========================================================
# CONFIG
# =========================================================
DOCX_PATH = Path(r"D:\Experiments\Hiberno_English_Dictonary\data\A Dictionary of Hiberno-English - Terence Patrick Dolan_Semi_clensed.docx")
BASE_DIR = Path(r"D:\Experiments\Hiberno_English_Dictonary\data")

AUDIT_DIR = BASE_DIR / "audit"
RUNS_DIR = BASE_DIR / "runs"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = RUNS_DIR / "run_registry.json"

PARSER_LABEL = "hiberno_lexical_parser"
PARSER_CODE_VERSION = "v12.0"
RUN_NOTES = "POS Pattern Update"

KEY_COL = "para_id"

In [3]:

COMPARE_COLS = [
    "entry_type",
    "headword_raw",
    "headword",
    "variant_forms_raw",
    "variant_forms",
    "pronunciation",
    "pronunciation_2",
    "pronunciation_3",
    "pronunciation_4",
    "part_of_speech",
    "grammatical_labels",
    "definition",
    "etymology",
    "cross_references",
    "examples",
    "region_mentions",
    "parse_confidence",
    "parse_notes",
]

# =========================================================
# RUN REGISTRY HELPERS
# =========================================================
def load_registry(path=REGISTRY_PATH):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "project": "hiberno_english_dictionary_parser",
        "created_at_utc": datetime.utcnow().isoformat(),
        "next_run_id": 1,
        "runs": []
    }

def save_registry(registry, path=REGISTRY_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(registry, f, ensure_ascii=False, indent=2)

def start_new_run(parser_label, parser_code_version, notes="", registry_path=REGISTRY_PATH):
    registry = load_registry(registry_path)

    run_id = registry["next_run_id"]
    run_name = f"run_{run_id:03d}"
    previous_run_id = registry["runs"][-1]["run_id"] if registry["runs"] else None

    run_meta = {
        "run_id": run_id,
        "run_name": run_name,
        "timestamp_utc": datetime.utcnow().isoformat(),
        "parser_label": parser_label,
        "parser_code_version": parser_code_version,
        "notes": notes,
        "previous_run_id": previous_run_id,
        "docx_path": str(DOCX_PATH),
        "audit_file": None,
        "parsed_file": None,
        "comparison_file": None,
        "n_raw_paragraphs": None,
        "n_clean_paragraphs": None,
        "n_records": None,
        "change_summary": None,
    }

    registry["next_run_id"] += 1
    registry["runs"].append(run_meta)
    save_registry(registry, registry_path)
    return run_meta

def update_run_record(run_id, updates, registry_path=REGISTRY_PATH):
    registry = load_registry(registry_path)
    for run in registry["runs"]:
        if run["run_id"] == run_id:
            run.update(updates)
            break
    save_registry(registry, registry_path)

def get_previous_run_meta(current_run_id, registry_path=REGISTRY_PATH):
    registry = load_registry(registry_path)
    current = next((r for r in registry["runs"] if r["run_id"] == current_run_id), None)
    if current is None:
        return None
    prev_id = current.get("previous_run_id")
    if prev_id is None:
        return None
    return next((r for r in registry["runs"] if r["run_id"] == prev_id), None)

def registry_as_df(path=REGISTRY_PATH):
    registry = load_registry(path)
    return pd.DataFrame(registry.get("runs", []))

# =========================================================
# GENERAL HELPERS
# =========================================================
def canonical_value(v):
    if isinstance(v, float) and pd.isna(v):
        return None
    if v is None:
        return None
    if isinstance(v, list):
        return [canonical_value(x) for x in v]
    if isinstance(v, dict):
        return {k: canonical_value(v[k]) for k in sorted(v)}
    return v

def hash_text(text):
    text = "" if pd.isna(text) else str(text)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def make_parse_payload(row, compare_cols=COMPARE_COLS):
    return {col: canonical_value(row.get(col, None)) for col in compare_cols}

def make_parse_hash(row, compare_cols=COMPARE_COLS):
    payload = json.dumps(
        make_parse_payload(row, compare_cols=compare_cols),
        ensure_ascii=False,
        sort_keys=True
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

# =========================================================
# DOCX EXTRACTION
# =========================================================
def extract_paragraphs_from_docx(docx_path):
    doc = Document(docx_path)
    paragraphs = []
    for i, p in enumerate(doc.paragraphs, start=1):
        txt = p.text.strip()
        if txt:
            paragraphs.append({
                "para_id": i,
                "text": txt,
            })
    return pd.DataFrame(paragraphs)

# =========================================================
# AUDIT TABLE
# =========================================================
def build_audit_df(raw_df):
    df = raw_df.copy()

    df["char_count"] = df["text"].str.len()
    df["word_count"] = df["text"].str.split().str.len()
    df["line_count_est"] = df["text"].str.count(r"\n") + 1
    df["starts_with"] = df["text"].str[:20]
    df["ends_with"] = df["text"].str[-20:]
    df["has_digit"] = df["text"].str.contains(r"\d", regex=True)
    df["has_brackets"] = df["text"].str.contains(r"[\[\]\(\)]", regex=True)
    df["has_slash"] = df["text"].str.contains("/", regex=False)
    df["has_semicolon"] = df["text"].str.contains(";", regex=False)
    df["has_colon"] = df["text"].str.contains(":", regex=False)
    df["has_quote"] = df["text"].str.contains(r"[\"'“”‘’]", regex=True)
    df["non_ascii"] = df["text"].apply(lambda s: any(ord(c) > 127 for c in s))
    df["text_hash"] = df["text"].apply(hash_text)

    df["is_duplicate"] = df.duplicated(subset=["text"], keep=False)
    df["duplicate_group"] = df.groupby("text")["para_id"].transform("min")

    df["flag_short"] = df["char_count"] < 25
    df["flag_long"] = df["char_count"] > 1200
    df["flag_low_word_count"] = df["word_count"] <= 3
    df["flag_possible_heading"] = (
        (df["word_count"] <= 6) &
        (~df["text"].str.contains(r"[.;:!?]", regex=True))
    )
    df["flag_single_letter_header"] = df["text"].str.strip().str.fullmatch(r"[A-Z]")
    df["flag_all_caps_short"] = (
        df["text"].str.fullmatch(r"[A-Z&\- ]{1,8}") &
        (df["word_count"] <= 2)
    ).fillna(False)

    flag_cols = [
        "is_duplicate",
        "flag_short",
        "flag_long",
        "flag_low_word_count",
        "flag_possible_heading",
        "flag_single_letter_header",
        "flag_all_caps_short",
    ]
    df["needs_review"] = df[flag_cols].any(axis=1)

    return df

def clean_audit_df(audit_df):
    df = audit_df.copy()
    df = df[~df["flag_single_letter_header"]].copy()
    df = df.reset_index(drop=True)
    return df

In [4]:
# =========================================================
# PARSER HELPERS
# =========================================================
POS_PAT = (
    r'(?:'
    r'personal pron\.|'
    r'reflexive pron\.|'
    r'pronominal phr\.|'
    r'pres\.part\.|'
    r'n\. phr\.|'
    r'n\.phr\.|'
    r'v\. phr\.|'
    r'v\.phr\.|'
    r'n\., v\.|'
    r'n\., adj\.|'
    r'n\. pl\.|'
    r'n\.pl\.|'
    r'v\. n\.|'
    r'v\.n\.|'
    r'interj\.|'
    r'exclam\.|'
    r'def\. art\.|'
    r'prep\.|'
    r'conj\.|'
    r'pron\.|'
    r'part\.|'
    r'int\.|'
    r'voc\.|'
    r'phr\.|'
    r'adj\.|'
    r'adv\.|'
    r'n\.|'
    r'v\.|'
    r'num\.|'
    r'excl\.'
    r')'
)

def normalize_entry_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'/\s*,\s*/', '/, /', text)
    text = re.sub(r'\s+([,.;:])', r'\1', text)
    return text

def extract_examples(text: str) -> list:
    return [e.strip() for e in re.findall(r'[“\"](.*?)[”\"]', text) if e.strip()]

def extract_region_mentions(text: str) -> list:
    return [
        {"code": m.group(1), "place": m.group(2)}
        for m in re.finditer(r'\(([A-Z&]{1,5}|[A-Z][A-Za-z]+),\s*([A-Z][A-Za-z .-]+)\)', text)
    ]

def split_definition_etymology(text: str):
    if not text:
        return None, None

    text = text.strip()

    if text.startswith("<"):
        return None, text.lstrip("<").strip(" .;")

    m = re.search(r'\s<\s*', text)
    if not m:
        return text.strip(" .;") or None, None

    left = text[:m.start()]
    right = text[m.end():]

    stop_positions = []
    for pat in [r'[“"]', r'\bSee\b']:
        m2 = re.search(pat, right)
        if m2:
            stop_positions.append(m2.start())

    if stop_positions:
        cut = min(stop_positions)
        etym = right[:cut].strip(" .;")
        tail = right[cut:].strip()
        definition = (left.strip(" .;") + ". " + tail).strip(" .;")
    else:
        etym = right.strip(" .;")
        definition = left.strip(" .;")

    return definition or None, etym or None

def split_cross_references(text: str) -> list:
    if not text:
        return []
    text = str(text).strip().rstrip(".")
    return [x.strip(" .;") for x in re.split(r'\s*;\s*', text) if x.strip(" .;")]

# =========================================================
# PARSER
# =========================================================
def parse_entry(text: str, idx: int) -> dict:
    text = normalize_entry_text(text)

    rec = {
        "para_id": idx,
        "id": f"hde_{idx:05d}",
        "source_text": text,
        "headword_raw": None,
        "headword": None,
        "variant_forms_raw": None,
        "variant_forms": [],
        "pronunciation": None,
        "pronunciation_2": None,
        "pronunciation_3": None,
        "pronunciation_4": None,
        "part_of_speech": None,
        "grammatical_labels": [],
        "definition": None,
        "etymology": None,
        "cross_references": [],
        "examples": extract_examples(text),
        "region_mentions": extract_region_mentions(text),
        "entry_type": "lexical",
        "parse_confidence": 0.5,
        "parse_notes": [],
    }

    # -----------------------------------------------------
    # PURE CROSS-REFERENCE ENTRIES
    # Must come before generic comma-definition parsing
    # and before pronunciation/POS parsing
    # -----------------------------------------------------

    # headword /pron/, see TARGET.
    m = re.match(
        r'^(?P<lemma>[^/]{1,140}?)\s*/(?P<pron>[^/]+?)/\s*,?\s*see\s+(?P<targets>.+?)(?:\.)?$',
        text,
        flags=re.I
    )
    if m:
        lemma = m.group("lemma").strip(" ,.")
        targets_raw = m.group("targets").strip()

        rec["entry_type"] = "cross_reference"
        rec["headword_raw"] = lemma
        rec["headword"] = lemma.lower()
        rec["pronunciation"] = m.group("pron").strip()
        rec["cross_references"] = split_cross_references(targets_raw)
        rec["definition"] = None
        rec["parse_confidence"] = 1.0
        rec["parse_notes"] = ["Parsed as cross-reference entry with pronunciation"]
        return rec

    # headword POS. see TARGET.
    m = re.match(
        rf'^(?P<lemma>[^/,.]{{1,140}}?)\s+(?P<pos>{POS_PAT})\s*see\s+(?P<targets>.+?)(?:\.)?$',
        text,
        flags=re.I
    )
    if m:
        lemma = m.group("lemma").strip(" ,.")
        targets_raw = m.group("targets").strip()

        rec["entry_type"] = "cross_reference"
        rec["headword_raw"] = lemma
        rec["headword"] = lemma.lower()
        rec["part_of_speech"] = m.group("pos").strip()
        rec["cross_references"] = split_cross_references(targets_raw)
        rec["definition"] = None
        rec["parse_confidence"] = 1.0
        rec["parse_notes"] = ["Parsed as cross-reference entry with POS"]
        return rec

    # headword, see TARGET.
    m = re.match(
        r'^(?P<lemma>[^,]{1,140}?)\s*,\s*see\s+(?P<targets>.+?)(?:\.)?$',
        text,
        flags=re.I
    )
    if m:
        lemma = m.group("lemma").strip(" ,.")
        targets_raw = m.group("targets").strip()

        if (
            not re.search(POS_PAT, lemma, flags=re.I)
            and "/" not in lemma
            and ":" not in lemma
            and "“" not in lemma
            and '"' not in lemma
        ):
            rec["entry_type"] = "cross_reference"
            rec["headword_raw"] = lemma
            rec["headword"] = lemma.lower()
            rec["cross_references"] = split_cross_references(targets_raw)
            rec["definition"] = None
            rec["parse_confidence"] = 1.0
            rec["parse_notes"] = ["Parsed as cross-reference entry"]
            return rec

    # headword, see.
    m = re.match(r'^(?P<lemma>.+?),\s*see\.?$', text, flags=re.I)
    if m:
        rec["entry_type"] = "cross_reference"
        rec["headword_raw"] = m.group("lemma").strip(" ,.")
        rec["headword"] = rec["headword_raw"].lower()
        rec["cross_references"] = []
        rec["parse_confidence"] = 0.6
        rec["parse_notes"] = ["Incomplete cross-reference target"]
        return rec

    # -----------------------------------------------------
    # PHRASE / EXPRESSION LABEL ENTRIES
    # Must come before generic comma-definition parsing
    # -----------------------------------------------------

    # headword /pron/ [POS], in the phrase/expression ...
    m = re.match(
        rf'''
        ^(?P<lemma>[^/]+?)
        \s*/(?P<pron>[^/]+)/
        (?:\s+(?P<pos>{POS_PAT}))?
        \s*,\s*
        (?P<label>in\ the\ phrase|in\ the\ expression)
        \s+(?P<rest>.+)$
        ''',
        text,
        flags=re.I | re.X
    )
    if m:
        lemma = m.group("lemma").strip()
        rec["headword_raw"] = lemma
        rec["headword"] = lemma.lower()
        rec["pronunciation"] = m.group("pron").strip()

        if m.group("pos"):
            rec["part_of_speech"] = m.group("pos").strip()

        rec["grammatical_labels"] = [m.group("label").strip().lower()]

        definition, etymology = split_definition_etymology(m.group("rest").strip())
        rec["definition"] = definition
        rec["etymology"] = etymology

        rec["cross_references"] += [
            x.strip(" .)")
            for x in re.findall(
                r'(?:^|[.;]\s+|\(\s*)(?:See|see)\s+([A-Z][A-Z0-9 \-’\'\.]+)',
                text
            )
        ]

        rec["parse_confidence"] = 0.96
        rec["parse_notes"] = ["Parsed as phrase/expression-labelled entry"]
        return rec

    # fallback: no pronunciation, but still phrase/expression-labelled
    m = re.match(
        r'^(?P<lemma>[^,]{1,140}),\s*(?P<label>in the phrase|in the expression)\s*(?P<rest>.+)$',
        text,
        flags=re.I
    )
    if m:
        rec["headword_raw"] = m.group("lemma").strip()
        rec["headword"] = rec["headword_raw"].lower()
        rec["grammatical_labels"] = [m.group("label").strip().lower()]

        definition, etymology = split_definition_etymology(m.group("rest").strip())
        rec["definition"] = definition
        rec["etymology"] = etymology

        rec["cross_references"] += [
            x.strip(" .)")
            for x in re.findall(
                r'(?:^|[.;]\s+|\(\s*)(?:See|see)\s+([A-Z][A-Z0-9 \-’\'\.]+)',
                text
            )
        ]

        rec["parse_confidence"] = 0.90
        rec["parse_notes"] = ["Parsed as phrase/expression-labelled entry"]
        return rec

    # -----------------------------------------------------
    # GRAMMATICAL / USAGE NOTE ENTRIES
    # Must come before generic comma-definition parsing
    # -----------------------------------------------------
    m = re.match(
        r'^(?P<lemma>[^,/]{1,140}?)\s+are commonly used\s+(?P<rest>.+)$',
        text,
        flags=re.I
    )
    if m:
        rec["entry_type"] = "grammatical_note"
        rec["headword_raw"] = m.group("lemma").strip(" ,.")
        rec["headword"] = rec["headword_raw"].lower()
        rec["grammatical_labels"] = ["are commonly used"]
        rec["definition"] = m.group("rest").strip(" ,.")
        rec["parse_confidence"] = 0.82
        rec["parse_notes"] = ["Parsed as grammatical-note entry"]
        return rec

    m = re.match(
        r'^(?P<lemma>[A-Z][A-Z0-9&.\'-]{1,20})\s+(?P<label>abbreviation for)\s+(?P<rest>.+)$',
        text,
        flags=re.I
    )
    if m:
        rec["headword_raw"] = m.group("lemma").strip()
        rec["headword"] = rec["headword_raw"].lower()
        rec["grammatical_labels"] = [m.group("label").strip().lower()]
        rec["definition"] = m.group("rest").strip(" ,.")
        rec["parse_confidence"] = 0.95
        return rec

    m = re.match(
        r'^(?P<lemma>[^/]{1,140}?)(?:\s*\((?P<label>[^)]*)\))?,\s*(?P<rest>.+)$',
        text
    )
    if m:
        lemma = m.group("lemma").strip()
        rest = m.group("rest").strip()
        if not re.match(rf'^{POS_PAT}\b', rest):
            rec["headword_raw"] = lemma
            rec["headword"] = lemma.lower()
            if m.group("label"):
                rec["grammatical_labels"].append(m.group("label").strip())
            definition, etymology = split_definition_etymology(rest)
            rec["definition"] = definition
            rec["etymology"] = etymology
            rec["parse_confidence"] = 0.83
            rec["parse_notes"] = ["Parsed as comma-definition entry"]
            return rec

    m = re.match(r'^(?P<lemma>.+?)\s*/(?P<pron>[^/]+?)\?\s+(?P<rest>' + POS_PAT + r'.+)$', text)
    if m:
        repaired = f"{m.group('lemma')} /{m.group('pron').strip()}/ {m.group('rest')}"
        rec2 = parse_entry(repaired, idx)
        rec2["parse_notes"].append("Recovered malformed pronunciation closer")
        rec2["parse_confidence"] = max(rec2["parse_confidence"], 0.78)
        return rec2

    m = re.match(r'^(?P<lemma>.+?)\s*/(?P<pron>[^/]+)/(?P<rest>.*)$', text)
    if m:
        left = m.group("lemma").strip()
        rest = m.group("rest").strip()

        rec["headword_raw"] = left

        hw = re.split(r'\s+also\s+|,\s*', left, maxsplit=1)[0].strip()
        rec["headword"] = hw.lower()

        if hw != left:
            rec["variant_forms_raw"] = left
            variants = [v.strip() for v in re.split(r'\s+also\s+|,\s*', left) if v.strip()]
            rec["variant_forms"] = [v for v in variants if v.lower() != rec["headword"]]

        rec["pronunciation"] = m.group("pron").strip()

        extra_prons = []
        rest_for_prons = rest
        while True:
            m_extra = re.match(r'^\s*,\s*/([^/]+)/(?P<tail>.*)$', rest_for_prons)
            if not m_extra:
                break
            extra_prons.append(m_extra.group(1).strip())
            rest_for_prons = m_extra.group("tail").strip()

        if len(extra_prons) > 0:
            rec["pronunciation_2"] = extra_prons[0]
        if len(extra_prons) > 1:
            rec["pronunciation_3"] = extra_prons[1]
        if len(extra_prons) > 2:
            rec["pronunciation_4"] = extra_prons[2]

        rest = rest_for_prons.strip(" ,;")

        variant_prefix_match = re.match(
            rf'^(?P<variant_prefix>(?:also|sometimes also)\s+.*?)(?=\s+{POS_PAT}(?:\s|,|$))',
            rest,
            flags=re.I
        )
        if variant_prefix_match:
            variant_raw = variant_prefix_match.group("variant_prefix").strip()
            cleaned_variant_raw = re.sub(r'^(?:also|sometimes also)\s+', '', variant_raw, flags=re.I).strip()
            cleaned_variant_raw = re.sub(r'/[^/]+/', '', cleaned_variant_raw).strip(" ,;")
            cleaned_variant_raw = re.sub(r'\betc\.$', '', cleaned_variant_raw, flags=re.I).strip(" ,;")

            extra_variants = [
                v.strip(" .;,")
                for v in re.split(r',\s*|\s+or\s+', cleaned_variant_raw)
                if v.strip(" .;,")
            ]

            rec["variant_forms_raw"] = (
                f"{rec['variant_forms_raw']}; {variant_raw}"
                if rec["variant_forms_raw"] else variant_raw
            )

            for v in extra_variants:
                if v and v.lower() != rec["headword"] and v not in rec["variant_forms"]:
                    rec["variant_forms"].append(v)

            rest = rest[variant_prefix_match.end():].lstrip(" ,;")

        sense_lead = None
        m_sense = re.match(r'^(?P<sense>\d+\.)\s*(?P<after>.+)$', rest)
        if m_sense:
            sense_lead = m_sense.group("sense")
            rest = m_sense.group("after").strip()

        rest = re.sub(
            r'^(adj|adv|prep|conj|pron|part|int|interj|voc|phr|n|v|num|excl|exclam)\s*,',
            r'\1.,',
            rest,
            flags=re.I
        )

        pm = re.match(rf'^(?P<pos>{POS_PAT})(?:\s*\((?P<label>[^)]*)\))?(?P<after>.*)$', rest)
        if pm:
            rec["part_of_speech"] = pm.group("pos").strip()
            if pm.group("label"):
                rec["grammatical_labels"].append(pm.group("label").strip())
            after = pm.group("after").lstrip(" ,")
            rec["parse_confidence"] = 0.90
        else:
            after = rest
            rec["parse_notes"].append("POS not confidently parsed")
            rec["parse_confidence"] = 0.72

        definition, etymology = split_definition_etymology(after)
        if definition and sense_lead:
            definition = f"{sense_lead} {definition}"

        rec["definition"] = definition
        rec["etymology"] = etymology
        rec["cross_references"] += [
            x.strip(" .)")
            for x in re.findall(
                r'(?:^|[.;]\s+|\(\s*)(?:See|see)\s+([A-Z][A-Z0-9 \-’\'\.]+)',
                text
            )
        ]
        return rec

    m = re.match(r'^(?P<lemma>[^/]{1,120}?)\s+(?P<rest>' + POS_PAT + r'.+)$', text)
    if m:
        rec["headword_raw"] = m.group("lemma").strip()
        rec["headword"] = rec["headword_raw"].lower()
        rest = m.group("rest").strip()

        rest = re.sub(
            r'^(adj|adv|prep|conj|pron|part|int|interj|voc|phr|n|v|num|excl|exclam)\s*,',
            r'\1.,',
            rest,
            flags=re.I
        )

        pm = re.match(rf'^(?P<pos>{POS_PAT})(?:\s*\((?P<label>[^)]*)\))?(?P<after>.*)$', rest)
        if pm:
            rec["part_of_speech"] = pm.group("pos").strip()
            if pm.group("label"):
                rec["grammatical_labels"].append(pm.group("label").strip())

            definition, etymology = split_definition_etymology(pm.group("after").lstrip(" ,"))
            rec["definition"] = definition
            rec["etymology"] = etymology
            rec["parse_confidence"] = 0.78
            return rec

    m = re.match(
        r'^(?P<lemma>[^/]{1,120}?)\s+also\s+(?P<variant>[^/]{1,120}?)\s+(?P<rest>.+)$',
        text,
        flags=re.I
    )
    if m:
        rec["headword_raw"] = m.group("lemma").strip()
        rec["headword"] = rec["headword_raw"].lower()
        rec["variant_forms_raw"] = f"{m.group('lemma').strip()} also {m.group('variant').strip()}"
        rec["variant_forms"] = [m.group("variant").strip()]
        definition, etymology = split_definition_etymology(m.group("rest").strip())
        rec["definition"] = definition
        rec["etymology"] = etymology
        rec["parse_confidence"] = 0.74
        rec["parse_notes"] = ["Parsed as variant-without-pronunciation entry"]
        return rec

    m = re.match(r'^(?P<lemma>.+?)\s+(?P<pron>[^ ]+/)\s*(?P<rest>.+)$', text)
    if m:
        repaired = f"{m.group('lemma')} /{m.group('pron').rstrip('/').strip()}/ {m.group('rest')}"
        rec2 = parse_entry(repaired, idx)
        rec2["parse_notes"].append("Recovered malformed pronunciation delimiter")
        rec2["parse_confidence"] = max(rec2["parse_confidence"], 0.75)
        return rec2

    rec["entry_type"] = "unparsed"
    rec["headword_raw"] = text.split(" ", 1)[0]
    rec["headword"] = rec["headword_raw"].lower()
    rec["definition"] = text
    rec["parse_notes"].append("Could not confidently parse entry structure")
    rec["parse_confidence"] = 0.2
    return rec

In [5]:
# =========================================================
# PARSED DF + COMPARISON
# =========================================================
def build_parsed_df(records):
    df = pd.DataFrame(records).copy()

    if KEY_COL not in df.columns:
        raise ValueError(f"Missing required key column: {KEY_COL}")
    if "source_text" not in df.columns:
        raise ValueError("Missing required column: source_text")

    for col in COMPARE_COLS:
        if col not in df.columns:
            df[col] = None

    df["source_hash"] = df["source_text"].apply(hash_text)
    df["parse_hash"] = df.apply(make_parse_hash, axis=1)

    front = [KEY_COL]
    if "id" in df.columns:
        front.append("id")
    front += ["source_text", "source_hash", "parse_hash"]
    front += COMPARE_COLS

    front = [c for c in front if c in df.columns]
    rest = [c for c in df.columns if c not in front]

    return df[front + rest].copy()

def compare_runs(old_df, new_df, key_col=KEY_COL, compare_cols=COMPARE_COLS):
    old = old_df.copy()
    new = new_df.copy()

    if "source_hash" not in old.columns:
        old["source_hash"] = old["source_text"].apply(hash_text)
    if "parse_hash" not in old.columns:
        old["parse_hash"] = old.apply(make_parse_hash, axis=1)

    if "source_hash" not in new.columns:
        new["source_hash"] = new["source_text"].apply(hash_text)
    if "parse_hash" not in new.columns:
        new["parse_hash"] = new.apply(make_parse_hash, axis=1)

    keep_cols = [key_col, "source_text", "source_hash", "parse_hash"] + compare_cols
    old = old[[c for c in keep_cols if c in old.columns]]
    new = new[[c for c in keep_cols if c in new.columns]]

    merged = old.merge(
        new,
        on=key_col,
        how="outer",
        suffixes=("_old", "_new"),
        indicator=True
    )

    def classify(row):
        if row["_merge"] == "left_only":
            return "removed"
        if row["_merge"] == "right_only":
            return "added"
        if row["parse_hash_old"] == row["parse_hash_new"]:
            return "unchanged"
        return "changed"

    merged["change_status"] = merged.apply(classify, axis=1)

    def changed_columns(row):
        if row["_merge"] != "both":
            return []
        out = []
        for col in compare_cols:
            old_val = canonical_value(row.get(f"{col}_old", None))
            new_val = canonical_value(row.get(f"{col}_new", None))
            if old_val != new_val:
                out.append(col)
        return out

    merged["changed_columns"] = merged.apply(changed_columns, axis=1)
    merged["num_changed_columns"] = merged["changed_columns"].apply(len)
    merged["source_changed"] = merged.apply(
        lambda r: False if r["_merge"] != "both" else r["source_hash_old"] != r["source_hash_new"],
        axis=1
    )
    merged["parse_changed"] = merged["change_status"] == "changed"

    return merged



In [6]:
# =========================================================
# MAIN PIPELINE
# =========================================================
run_meta = start_new_run(
    parser_label=PARSER_LABEL,
    parser_code_version=PARSER_CODE_VERSION,
    notes=RUN_NOTES
)

RUN_ID = run_meta["run_id"]
RUN_NAME = run_meta["run_name"]

print(f"Starting {RUN_NAME}")

raw_paragraph_df = extract_paragraphs_from_docx(DOCX_PATH)

audit_df = build_audit_df(raw_paragraph_df)

# REMOVE HEADER ROWS HERE 👇
clean_df = audit_df[~audit_df["flag_single_letter_header"]].copy()



audit_file = AUDIT_DIR / f"paragraph_audit_{RUN_NAME}.xlsx"
with pd.ExcelWriter(audit_file, engine="openpyxl") as writer:
    audit_df.to_excel(writer, index=False, sheet_name="audit_raw")
    clean_df.to_excel(writer, index=False, sheet_name="clean_after_header_removal")
    audit_df[audit_df["needs_review"]].to_excel(writer, index=False, sheet_name="needs_review")

update_run_record(
    RUN_ID,
    {
        "audit_file": str(audit_file),
        "n_raw_paragraphs": int(len(raw_paragraph_df)),
        "n_clean_paragraphs": int(len(clean_df)),
    }
)

records = [
    parse_entry(text, int(para_id))
    for para_id, text in zip(clean_df["para_id"], clean_df["text"])
]
records = [r for r in records if r.get("entry_type") != "skip"]

parsed_df = build_parsed_df(records)

parsed_file = RUNS_DIR / f"parsed_output_{RUN_NAME}.xlsx"
with pd.ExcelWriter(parsed_file, engine="openpyxl") as writer:
    parsed_df.to_excel(writer, index=False, sheet_name="parsed_output")

jsonl_file = RUNS_DIR / f"parsed_output_{RUN_NAME}.jsonl"

with open(jsonl_file, "w", encoding="utf-8") as f:
    for _, row in parsed_df.iterrows():
        record = row.to_dict()
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


update_run_record(
    RUN_ID,
    {
        "parsed_file": str(parsed_file),
        "n_records": int(len(parsed_df)),
    }
)

print(f"Saved audit:  {audit_file}")
print(f"Saved parse:  {parsed_file}")

entry_type_counts = collections.Counter(parsed_df["entry_type"])
print("\nEntry type counts:")
print(dict(entry_type_counts))

prev_meta = get_previous_run_meta(RUN_ID)

if prev_meta and prev_meta.get("parsed_file"):
    prev_file = Path(prev_meta["parsed_file"])

    if prev_file.exists():
        old_df = pd.read_excel(prev_file)
        comparison_df = compare_runs(old_df, parsed_df)

        comparison_file = RUNS_DIR / f"comparison_{prev_meta['run_name']}_vs_{RUN_NAME}.xlsx"
        with pd.ExcelWriter(comparison_file, engine="openpyxl") as writer:
            comparison_df.to_excel(writer, index=False, sheet_name="comparison")
            comparison_df[comparison_df["change_status"] == "changed"].to_excel(
                writer, index=False, sheet_name="changed_only"
            )
            comparison_df[comparison_df["change_status"] == "unchanged"].to_excel(
                writer, index=False, sheet_name="unchanged_only"
            )
            comparison_df[comparison_df["change_status"] == "added"].to_excel(
                writer, index=False, sheet_name="added_only"
            )
            comparison_df[comparison_df["change_status"] == "removed"].to_excel(
                writer, index=False, sheet_name="removed_only"
            )

        change_summary = comparison_df["change_status"].value_counts(dropna=False).to_dict()
        update_run_record(
            RUN_ID,
            {
                "comparison_file": str(comparison_file),
                "change_summary": change_summary,
            }
        )

        print(f"\nCompared with previous run: {prev_meta['run_name']}")
        print(f"Saved comparison: {comparison_file}")
        print("Change summary:")
        print(change_summary)

        display_cols = [
            "para_id",
            "change_status",
            "source_changed",
            "parse_changed",
            "num_changed_columns",
            "changed_columns",
            "source_text_old",
            "source_text_new",
        ]
        print("\nSample changed rows:")
        display(comparison_df.loc[comparison_df["change_status"] == "changed", display_cols].head(20))
    else:
        print(f"\nPrevious parsed file not found: {prev_file}")
else:
    print("\nNo previous run available yet.")

print("\nRun registry:")
display(registry_as_df())

Starting run_009
Saved audit:  D:\Experiments\Hiberno_English_Dictonary\data\audit\paragraph_audit_run_009.xlsx
Saved parse:  D:\Experiments\Hiberno_English_Dictonary\data\runs\parsed_output_run_009.xlsx

Entry type counts:
{'lexical': 3534, 'cross_reference': 356, 'unparsed': 7, 'grammatical_note': 1}

Compared with previous run: run_008
Saved comparison: D:\Experiments\Hiberno_English_Dictonary\data\runs\comparison_run_008_vs_run_009.xlsx
Change summary:
{'changed': 3897, 'removed': 2, 'added': 1}

Sample changed rows:


,para_id,change_status,source_changed,parse_changed,num_changed_columns,changed_columns,source_text_old,source_text_new
1,1,changed,True,True,10,"[variant_forms, part_of_speech, grammatical_la...",a /ə/ voc. particle (used when speaking to som...,a /ə/ indefinite article. The absence of an in...
2,2,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","ABCs /eːbiːsiːz/ n. (colloq.), irregular red l...",a /ə/ voc. particle (used when speaking to som...
3,3,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","abhaile /əˈwɑljə/ adv., home, homewards < Ir. ...","ABCs /eːbiːsiːz/ n. (colloq.), irregular red l..."
4,4,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","abhainnín /auˈniːn/ n., small river, tributary...","abhaile /əˈwɑljə/ adv., home, homewards < Ir. ..."
5,5,changed,True,True,11,"[headword_raw, headword, variant_forms, pronun...","ábhairín /ɔːˈvriːn/ n., a small quantity, espe...","abhainnín /auˈniːn/ n., small river, tributary..."
6,6,changed,True,True,11,"[headword_raw, headword, variant_forms, pronun...","ábhar /ˈɔːvər/ n. 1. Appreciable quantity, fai...","ábhairín /ɔːˈvriːn/ n., a small quantity, espe..."
7,7,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","able /ˈeːbəl/ adj., adequate, fit to cope with...","ábhar /ˈɔːvər/ n. 1. Appreciable quantity, fai..."
8,8,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","aboo /əˈbuː/ int., for ever! to victory! < Ir ...","able /ˈeːbəl/ adj., adequate, fit to cope with..."
9,9,changed,True,True,12,"[headword_raw, headword, variant_forms, pronun...","above /əˈbʌv/ adv., loosely meaning the same a...","aboo /əˈbuː/ int., for ever! to victory! < Ir ..."
10,10,changed,True,True,11,"[headword_raw, headword, variant_forms, pronun...","abroad /əˈbrɔːd/ adv., outside (DOH, Limerick)...","above /əˈbʌv/ adv., loosely meaning the same a..."



Run registry:


,run_id,run_name,timestamp_utc,parser_label,parser_code_version,notes,previous_run_id,docx_path,audit_file,parsed_file,comparison_file,n_raw_paragraphs,n_clean_paragraphs,n_records,change_summary
0,1,run_001,2026-03-21T10:59:04.860820,hiberno_lexical_parser,v1.0,Initial single-notebook tracked pipeline,NaN,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,NaN,3899,3898,3898,None
1,2,run_002,2026-03-21T11:34:28.487049,hiberno_lexical_parser,v8.0,Initial single-notebook tracked pipeline,1.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'unchanged': 3524, 'changed': 374}"
2,3,run_003,2026-03-21T11:41:49.917540,hiberno_lexical_parser,v9.0,Initial single-notebook tracked pipeline,2.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'unchanged': 3895, 'changed': 3}"
3,4,run_004,2026-03-21T11:57:52.239311,hiberno_lexical_parser,v10.0,Initial single-notebook tracked pipeline,3.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'unchanged': 3851, 'changed': 47}"
4,5,run_005,2026-03-21T12:02:50.755246,hiberno_lexical_parser,v11.0,POS Pattern Update,4.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'unchanged': 3893, 'changed': 5}"
5,6,run_006,2026-03-21T12:17:01.924720,hiberno_lexical_parser,v12.0,POS Pattern Update,5.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,{'unchanged': 3898}
6,8,run_008,2026-03-21T12:48:13.106746,hiberno_lexical_parser,v12.0,POS Pattern Update,6.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'unchanged': 3897, 'changed': 1}"
7,9,run_009,2026-03-27T22:33:49.255239,hiberno_lexical_parser,v12.0,POS Pattern Update,8.0,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,D:\Experiments\Hiberno_English_Dictonary\data\...,3899,3898,3898,"{'changed': 3897, 'removed': 2, 'added': 1}"
